In [ ]:
"""
Herbarium Processor Notebook
========================

OCR-free comparison notebook. Its sections follow herbarium_processor.ipynb:
configure inputs, set prompt instructions, prepare images, build targets, then
run the model and save the results.

Set image_path to the same already-cropped image directory used for the
OCR-assisted run when comparing model results.
"""

from utils.notebook_setup import setup_project

setup_project()

image_path = "data/img/bucket"
output_csv_path = "tmp/llm_only_results.csv"
config_path = "prompts/configs/ai_only_default_prompt.yaml"
system_instructions_path = "prompts/ai_only_system_instructions.md"
provider = "openrouter"  # choose "google" to use Google Gemini
model_name = None  # leave None to use the provider default

In [ ]:
"""
Configuration: loading the AI-only prompt settings and system instructions.

This uses the field list and few-shot examples from ai_only_default_prompt.yaml.
The OCR-based Jinja template is intentionally not loaded.
"""

import json
import math
import re

import pandas as pd
import yaml

from herbarium_processor.config import ROOT_DIR

with open(config_path, encoding="utf-8") as config_file:
    prompt_config = yaml.safe_load(config_file)

field_list = [str(field).strip() for field in prompt_config["field_list"]]
shot_config = prompt_config.get("shot_data", [])
labels_path = ROOT_DIR / prompt_config["csv_path"]
labels = pd.read_csv(labels_path, dtype={"id": str})
labels.columns = [str(column).strip() for column in labels.columns]
system_instructions = (ROOT_DIR / system_instructions_path).read_text(encoding="utf-8")


def json_safe(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


def load_example_output(example_id):
    matches = labels[labels["id"].astype(str) == str(example_id)]
    if matches.empty:
        raise ValueError(
            f"Few-shot example ID {example_id!r} is missing from {labels_path}"
        )
    row = matches.iloc[0].to_dict()
    return {field: json_safe(row.get(field)) for field in field_list}


def build_prompt(examples, target_name):
    listed_fields = "\n".join(f"- {field}" for field in field_list)
    sections = [
        "Extract the requested fields from the herbarium specimen label image.",
        "Return one JSON object with exactly these case-sensitive keys:",
        listed_fields,
        "Use the spellings above exactly, including Darwin Core camelCase; do not convert them to snake_case.",
        "Use the examples to match the field meanings and output format.",
    ]
    for index, (name, output) in enumerate(examples):
        sections.extend(
            [
                f"\nExample {index + 1}: {name}",
                f"Image: <|image_{index}|>",
                "Correct output:",
                json.dumps(output, ensure_ascii=False, indent=2),
            ]
        )
    sections.extend(
        [
            f"\nNow extract the fields from this image ({target_name}):",
            f"Image: <|image_{len(examples)}|>",
            "Return only one valid JSON object. Use null when a value is absent or illegible. Do not add keys or explanation.",
        ]
    )
    return "\n".join(sections)


def parse_model_json(raw_text):
    text = re.sub(r"^\s*```(?:json)?\s*|\s*```$", "", raw_text.strip())
    result = json.loads(text)
    if not isinstance(result, dict):
        raise ValueError("Model response must be a JSON object")
    return {field: result.get(field) for field in field_list}


print(f"Loaded {len(field_list)} fields and {len(shot_config)} few-shot examples")
print(f"System instructions: {system_instructions_path}")

In [ ]:
"""
Step 1: Image preparation.

This mirrors herbarium_processor.ipynb by converting HEIC files and applying
EXIF rotation/standard resizing. It makes no OCR calls. If image_path already
contains the exact processed images used for the OCR run, skip this cell.
"""

from herbarium_processor.core.image.image_utils import process_directory_no_auto_rotate

process_directory_no_auto_rotate(image_path)

In [ ]:
"""
Step 2: Building the few-shot examples and target image list.

This section uses image bytes and the known example outputs only. It does not
look for OCR files or construct SpecimenLabel objects that require OCR paths.
"""

IMAGE_MIME_TYPES = {
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".png": "image/png",
    ".webp": "image/webp",
}

few_shot_images = []
few_shot_examples = []
for shot in shot_config:
    shot_image = ROOT_DIR / shot["img_path"]
    if not shot_image.is_file():
        raise FileNotFoundError(f"Few-shot image not found: {shot_image}")
    few_shot_images.append(shot_image)
    few_shot_examples.append((shot_image.name, load_example_output(shot["id"])))

image_dir = ROOT_DIR / image_path
target_images = sorted(
    path
    for path in image_dir.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_MIME_TYPES
)
if not target_images:
    raise FileNotFoundError(f"No supported specimen images found in {image_dir}")

print(f"Prepared {len(few_shot_examples)} few-shot examples")
print(f"Found {len(target_images)} target images in {image_dir}")

In [ ]:
"""
Step 3: Previewing the OCR-free prompt.

The placeholders below are replaced with the few-shot images and target image
when each request is assembled.
"""

preview_prompt = build_prompt(few_shot_examples, target_images[0].name)
print(preview_prompt)

In [ ]:
"""
Step 4: Calling the vision model and saving the AI-only results.

No OCR client or OCR output is used in this section.
"""

import csv

from herbarium_processor.core.inference.llm_api import GoogleGeminiAPI, OpenRouterAPI

api_class = GoogleGeminiAPI if provider == "google" else OpenRouterAPI
llm = api_class(system_instructions=system_instructions, model_name=model_name)


def build_contents(target_image):
    prompt = build_prompt(few_shot_examples, target_image.name)
    image_paths = [*few_shot_images, target_image]
    parts = []
    cursor = 0
    for match in re.finditer(r"<\|image_(\d+)\|>", prompt):
        if match.start() > cursor:
            parts.append(prompt[cursor : match.start()])
        image_index = int(match.group(1))
        image_file = image_paths[image_index]
        parts.append(
            {
                "data": image_file.read_bytes(),
                "mime_type": IMAGE_MIME_TYPES[image_file.suffix.lower()],
            }
        )
        cursor = match.end()
    if cursor < len(prompt):
        parts.append(prompt[cursor:])
    return parts


async def run_ai_only_extraction():
    rows = []
    for image_file in target_images:
        print(f"Processing {image_file.name}")
        response = await llm.generate_content(build_contents(image_file))
        row = parse_model_json(response)
        row["id"] = image_file.stem
        rows.append(row)

    output_path = ROOT_DIR / output_csv_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=["id", *field_list])
        writer.writeheader()
        writer.writerows(rows)
    return pd.DataFrame(rows, columns=["id", *field_list]), output_path


results_df, results_path = await run_ai_only_extraction()
print(f"Saved {len(results_df)} records to {results_path}")
display(results_df.head())